In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

INPUT_PATH = INPUT_DIR / "adjusted_productivity_imported.csv"

LONG_PATH = OUTPUT_DIR / "productivity_long.csv"
TRANSITIONS_PATH = OUTPUT_DIR / "productivity_transitions.csv"
COMPLETE_PATH = OUTPUT_DIR / "productivity_complete.csv"
TRAJECTORIES_PATH = OUTPUT_DIR / "trajectories_complete.npy"
SUMMARY_PATH = OUTPUT_DIR / "preparation_summary.csv"
COUNTS_PATH = OUTPUT_DIR / "counts_by_year.csv"
EXCLUSIONS_PATH = OUTPUT_DIR / "preparation_exclusions.csv"
MANIFEST_PATH = OUTPUT_DIR / "preparation_manifest.json"

Y = 20
OBS_YEARS = np.arange(Y + 1)
TRANSITION_YEARS = np.arange(Y)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
assert INPUT_PATH.exists(), (f"Input file not found or symlink is broken: {INPUT_PATH}")

df_imported = pd.read_csv(INPUT_PATH,dtype={"dblp": "string"},)

required_cols = ["dblp", "phd_year", "CareerAge", "pubs_adj"]
missing_cols = sorted(set(required_cols) - set(df_imported.columns))

assert not missing_cols, f"Missing required columns: {missing_cols}"
assert len(df_imported) > 0, "Imported dataset is empty"

print(f"Imported shape: {df_imported.shape}")
display(df_imported.head())

Imported shape: (54948, 21)


,contribs,contribs_with_corr,current,dblp,department,facultyName,first_asst_job_rank,first_asst_job_year,has_postdoc,is_female,...,phd_rank,phd_year,place,pubs,pubs_adj,pubs_with_corr,pubs_with_corr_adj,recordDate,year,CareerAge
0,0.342857,0.342857,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,2,3.902057,2,3.902057,6/6/11,1998,-3
1,0.250000,0.250000,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,1,1.875160,1,1.875160,6/6/11,1999,-2
2,1.833333,1.833333,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,4,7.214900,4,7.214900,6/6/11,2000,-1
3,1.583333,1.583333,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,3,5.209137,3,5.209137,6/6/11,2001,0
4,1.444444,1.444444,Associate Professor,=Ccedil=etintemel:Ugur,Computer Science,Ugur Cetintemel,21.92,2001,False,False,...,28.65,2001.0,Brown University,4,6.691238,4,6.691238,6/6/11,2002,1


In [3]:
df = df_imported.copy()

df["dblp"] = df["dblp"].astype("string").str.strip()
df.loc[df["dblp"].eq(""), "dblp"] = pd.NA

for col in ["phd_year", "CareerAge", "pubs_adj"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["phd_year", "CareerAge", "pubs_adj"]] = (df[["phd_year", "CareerAge", "pubs_adj"]].replace([np.inf, -np.inf], np.nan))


In [4]:
essential_cols = ["dblp", "phd_year", "CareerAge", "pubs_adj"]

missing_essential = df[essential_cols].isna().any(axis=1)
negative_productivity = df["pubs_adj"].lt(0).fillna(False)
outside_career_window = (~df["CareerAge"].between(0, Y) & df["CareerAge"].notna())

exclusion_rows = pd.DataFrame([
    {"reason": "missing_essential_field","rows": int(missing_essential.sum()),},
    {"reason": "negative_productivity","rows": int((~missing_essential & negative_productivity).sum()),},
    {"reason": "outside_career_window","rows": int((~missing_essential & ~negative_productivity & outside_career_window).sum()),},])

keep = (~missing_essential & ~negative_productivity & ~outside_career_window)

productivity_long = df.loc[keep].copy()

productivity_long["phd_year"] = productivity_long["phd_year"].astype("int64")
productivity_long["CareerAge"] = productivity_long["CareerAge"].astype("int64")
productivity_long["pubs_adj"] = productivity_long["pubs_adj"].astype("float64")

In [5]:
productivity_long["dblp_id"] = (productivity_long["dblp"].astype("string") + "__" + productivity_long["phd_year"].astype("string"))

In [6]:
transition_source = productivity_long.copy()

transition_source["CareerAge_next"] = (transition_source.groupby("dblp_id")["CareerAge"].shift(-1))

transition_source["pubs_adj_next"] = (transition_source.groupby("dblp_id")["pubs_adj"].shift(-1))

is_consecutive = transition_source["CareerAge_next"].eq(transition_source["CareerAge"] + 1)

is_valid_start = transition_source["CareerAge"].between(0, Y - 1)

productivity_transitions = (transition_source.loc[is_consecutive & is_valid_start,["dblp_id","dblp","phd_year","CareerAge","CareerAge_next","pubs_adj","pubs_adj_next"]].copy().reset_index(drop=True))

productivity_transitions["CareerAge_next"] = (productivity_transitions["CareerAge_next"].astype("int64"))

productivity_transitions["raw_delta"] = (productivity_transitions["pubs_adj_next"]- productivity_transitions["pubs_adj"])

assert productivity_transitions["CareerAge_next"].eq(productivity_transitions["CareerAge"] + 1).all()

In [7]:

productivity_panel = (productivity_long.pivot(index="dblp_id",columns="CareerAge",values="pubs_adj").reindex(columns=OBS_YEARS))

productivity_complete = productivity_panel.dropna().copy()

assert productivity_complete.shape[1] == Y + 1
assert productivity_complete.notna().all().all()

trajectories_complete = productivity_complete.to_numpy(dtype=float).T

assert trajectories_complete.shape[0] == Y + 1
assert np.isfinite(trajectories_complete).all()

print(f"complete careers: {len(productivity_complete):,}")
print(f"traj arr shape: {trajectories_complete.shape}")

complete careers: 681
traj arr shape: (21, 681)


In [8]:
# Cell 9: save prepared datasets and reports

productivity_long.to_csv(LONG_PATH, index=False)
productivity_transitions.to_csv(TRANSITIONS_PATH, index=False)
productivity_complete.reset_index().to_csv(COMPLETE_PATH, index=False)
np.save(TRAJECTORIES_PATH, trajectories_complete)